# 01 — Top-down pipeline only

Questo notebook esegue **solo la prima parte/top-down** della pipeline GORE. La generazione di description, actors, HLG e LLG usa i generatori originali; non viene eseguito il ciclo bottom-up.

Configurazione sperimentale: **Actors, HLG e LLG Zero-shot con critic Few-shot abilitato**. Il critic usa quattro esempi, soglia `8.5/10` e al massimo tre iterazioni. Gli LLG vengono generati e valutati globalmente a partire dall'intero insieme degli HLG.

Gli output strutturati vengono salvati in `output/top_down/` e possono essere riutilizzati dal notebook bottom-up e dal notebook di valutazione.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print("Project root:", PROJECT_ROOT)

import src.llm_clients as llm_clients
llm_clients.reset_model_clients()
print("Qwen locale:", llm_clients.QWEN_GENERATOR_MODEL)
print("Critic top-down:", llm_clients.configured_critic_model())
print("LM Studio:", llm_clients.LM_STUDIO_BASE_URL)


Project root: C:\Users\agnes\Desktop\TESI FINALE CON EXPERIMENT\LLM_MultiAgentChain_GoalOrientedAPIAlignement
Qwen locale: qwen/qwen3-vl-8b
Critic top-down: mistralai/ministral-3-14b-reasoning
LM Studio: http://127.0.0.1:1234/v1


## 1. Configurazione

La documentazione segue il percorso originale: se la ground truth contiene `link-readme`, viene chiamato `generate_description(...)`; altrimenti viene usata direttamente la descrizione disponibile nella ground truth. Non viene introdotto chunking aggiuntivo nella baseline.

La configurazione corrente seleziona soltanto **SIA Project 25 26** con `SELECTED_DATASETS = ["SIA Project 25 26"]` e `SKIP_COMPLETED_DATASETS = []`: il dataset viene rigenerato anche se il JSON esiste. Gli altri dataset non vengono eseguiti. Per selezionarli tutti, impostare `SELECTED_DATASETS = None`. Nomi sconosciuti o duplicati vengono rifiutati prima delle chiamate ai modelli.


In [2]:
import json

from groundtruth import ALL_GROUNDTRUTHS
from src.extraction.extractor import (
    generate_description,
    generate_actors,
    generate_high_level_goals,
    generate_low_level_goals,
)
from src.self_critique.refine_response import (
    EvalMode,
    generate_response_with_reflection,
)
import src.self_critique.refine_response as reflection
from src.examples.shot_learning import ShotPromptingMode
from src.utils import get_markdown
import src.llm_clients as llm_clients

# Rerun only this dataset, including when its baseline file already exists.
# Use None to select all datasets.
SELECTED_DATASETS = ["SIA Project 25 26"]
SKIP_COMPLETED_DATASETS = []


def select_top_down_datasets(projects, selected_names):
    projects = list(projects)
    if selected_names is None:
        return projects
    if isinstance(selected_names, str) or not selected_names:
        raise ValueError("SELECTED_DATASETS must be a non-empty list of names, or None for all datasets.")
    if len(set(selected_names)) != len(selected_names):
        raise ValueError("SELECTED_DATASETS contains duplicate dataset names.")
    by_name = {project["name"]: project for project in projects}
    unknown = [name for name in selected_names if name not in by_name]
    if unknown:
        raise ValueError(f"Unknown selected datasets: {unknown}")
    return [by_name[name] for name in selected_names]


GROUNDTRUTHS = select_top_down_datasets(ALL_GROUNDTRUTHS, SELECTED_DATASETS)
if SKIP_COMPLETED_DATASETS:
    select_top_down_datasets(ALL_GROUNDTRUTHS, SKIP_COMPLETED_DATASETS)
if not GROUNDTRUTHS:
    raise RuntimeError("No projects found in groundtruth.py")

ACTOR_PROMPTING_MODE = ShotPromptingMode.ZERO_SHOT
HLG_PROMPTING_MODE = ShotPromptingMode.ZERO_SHOT
LLG_PROMPTING_MODE = ShotPromptingMode.ZERO_SHOT
ACTOR_EVALUATOR_ABLATION = False
HLG_EVALUATOR_ABLATION = False
LLG_EVALUATOR_ABLATION = False
TOP_DOWN_MAX_ATTEMPTS = 3
TOP_DOWN_QUALITY_THRESHOLD = 8.5
TOP_DOWN_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT = 4
PIPELINE_PROFILE = "ALL_ZERO_SHOT_WITH_FEW_SHOT_CRITIC"
# Keep the established filename so a rerun replaces the stale baseline
# instead of creating duplicate datasets for the bottom-up notebook.
OUTPUT_FILE_PROFILE = "ZERO_SHOT_feedback_original_top_down"

OUTPUT_ROOT = PROJECT_ROOT / "output"
TOP_DOWN_DIR = OUTPUT_ROOT / "top_down"
TOP_DOWN_DIR.mkdir(parents=True, exist_ok=True)

def top_down_output_file_for(dataset_name: str) -> Path:
    return TOP_DOWN_DIR / f"{dataset_name}_{OUTPUT_FILE_PROFILE}_top_down.json"

def run_top_down_reflection(**kwargs):
    """Run the paper loop parameters without changing bottom-up defaults."""
    previous_threshold = reflection.QUALITY_THRESHOLD
    reflection.QUALITY_THRESHOLD = TOP_DOWN_QUALITY_THRESHOLD
    try:
        return generate_response_with_reflection(
            max_attempts=TOP_DOWN_MAX_ATTEMPTS,
            **kwargs,
        )
    finally:
        reflection.QUALITY_THRESHOLD = previous_threshold

print(f"Projects selected: {len(GROUNDTRUTHS)} / {len(ALL_GROUNDTRUTHS)}")
for project_index, project in enumerate(GROUNDTRUTHS, start=1):
    print(f"  {project_index}/{len(GROUNDTRUTHS)} - {project['name']}")
print("Skip if output already exists:", SKIP_COMPLETED_DATASETS)
print("Top-down output directory:", TOP_DOWN_DIR)
print("Qwen generator/reconstructor (LM Studio locale):", llm_clients.QWEN_GENERATOR_MODEL)
print("Qwen generator max output tokens:", llm_clients.QWEN_GENERATOR_MAX_TOKENS)
print("Ministral critic (LM Studio):", llm_clients.configured_critic_model())
print("LM Studio endpoint:", llm_clients.LM_STUDIO_BASE_URL)
print("Ministral critic max output tokens:", llm_clients.MINISTRAL_CRITIC_MAX_TOKENS)
print("Actor prompting:", ACTOR_PROMPTING_MODE.name, "| critic: ON (Few-shot)")
print("HLG prompting:", HLG_PROMPTING_MODE.name, "| critic: ON (Few-shot)")
print("LLG prompting:", LLG_PROMPTING_MODE.name, "| critic: ON (Few-shot)")
print("Top-down evaluator few-shot examples:", TOP_DOWN_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT)
print("Top-down quality threshold:", TOP_DOWN_QUALITY_THRESHOLD)
print("Top-down max reflection attempts:", TOP_DOWN_MAX_ATTEMPTS)



Projects selected: 12 / 12
  1/12 - SIA Project 25 26
  2/12 - Assegno Unico Universale - SIA Project 24 25
  3/12 - La Reine Marlene - SIA Project 23 24
  4/12 - Ethical Purchasing Group - SIA Project 22 23
  5/12 - Event Organization Portal - SIA Project 21 22
  6/12 - Calculator
  7/12 - Fitness Tracker
  8/12 - Genome Nexus
  9/12 - Gestao Hospital
  10/12 - London Ambulance Service
  11/12 - Planning Poker
  12/12 - Recycling
Skip if output already exists: ['Recycling']
Top-down output directory: C:\Users\agnes\Desktop\TESI FINALE CON EXPERIMENT\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\top_down
Qwen generator/reconstructor (LM Studio locale): qwen/qwen3-vl-8b
Qwen generator max output tokens: 4000
Ministral critic (LM Studio): mistralai/ministral-3-14b-reasoning
LM Studio endpoint: http://127.0.0.1:1234/v1
Ministral critic max output tokens: 4096
Actor prompting: ZERO_SHOT | critic: ON (Few-shot)
HLG prompting: ZERO_SHOT | critic: ON (Few-shot)
LLG prompting: ZERO_SHOT

## 2. Esecuzione top-down e salvataggio

Per ogni dataset viene costruita la `description` con il percorso originale. Actors, HLG e LLG partono da una generazione Zero-shot e vengono valutati dal critic Few-shot. Come nella pipeline originale, tutti gli HLG e tutti gli LLG vengono valutati globalmente. Il JSON conserva anche gli oggetti Pydantic strutturati necessari al successivo bottom-up. Gli output già visualizzati appartengono all'esecuzione precedente e vengono sostituiti rieseguendo le celle.


Vengono eseguiti e sovrascritti gli output dei dataset selezionati, salvo quelli elencati in `SKIP_COMPLETED_DATASETS` con JSON gia presente. Il riepilogo distingue completati, falliti e saltati. La rigenerazione usa la pipeline esistente e non garantisce da sola la correzione dei riferimenti.


In [3]:
generated_descriptions = {}
generated_actor_objects = {}
generated_hl_objects = {}
generated_ll_objects = {}
baseline_errors = {}
skipped_outputs = {}


def run_top_down_for_dataset(groundtruth: dict) -> None:
    dataset_name = groundtruth["name"]
    existing_output = top_down_output_file_for(dataset_name)
    if dataset_name in SKIP_COMPLETED_DATASETS and existing_output.is_file():
        skipped_outputs[dataset_name] = existing_output
        print(f"[{dataset_name}] SKIPPED: output already saved: {existing_output.name}")
        return

    try:
        # ---------------------------------------------------------------
        # 1. ORIGINAL documentation preprocessing
        # ---------------------------------------------------------------
        if "link-readme" in groundtruth:
            documentation_link = groundtruth["link-readme"]

            # Saved only for traceability; generate_description() below follows
            # the original repository path and reads the README itself.
            raw_description = get_markdown(documentation_link)
            generated_description = generate_description(documentation_link)
            description = generated_description.description.strip()
            preprocessing_info = {
                "chunked": False,
                "chunkCount": 1,
                "originalChars": len(raw_description),
                "processedChars": len(description),
                "strategy": "original_generate_description",
            }
        else:
            raw_description = str(groundtruth["description"])
            description = raw_description

            preprocessing_info = {
                "chunked": False,
                "chunkCount": 1,
                "originalChars": len(raw_description),
                "processedChars": len(description),
                "strategy": "groundtruth_description",
            }

        print(
            f"[{dataset_name}] Documentation ready | "
            f"raw={len(raw_description)} chars | "
            f"pipeline={len(description)} chars"
        )

        # ---------------------------------------------------------------
        # 2. ACTORS: Zero-shot generation + Few-shot critic/reflection
        # ---------------------------------------------------------------
        actors, actors_score, actors_critique, actors_attempts, actors_history = run_top_down_reflection(
            target_type="Actors",
            call_function=generate_actors,
            define_args=(description,),
            eval_mode=EvalMode.ACTORS,
            eval_args=(description,),
            shotPromptingMode=ACTOR_PROMPTING_MODE,
            llama_ablation=ACTOR_EVALUATOR_ABLATION,
        )

        # ---------------------------------------------------------------
        # 3. HLG: Zero-shot generation + Few-shot critic/reflection
        # ---------------------------------------------------------------
        high_level_goals, hl_score, hl_critique, hl_attempts, hl_history = (
            run_top_down_reflection(
                target_type="High Level Goals",
                call_function=generate_high_level_goals,
                define_args=(description, actors),
                eval_mode=EvalMode.HIGH_LEVEL,
                eval_args=(description, actors),
                shotPromptingMode=HLG_PROMPTING_MODE,
                llama_ablation=HLG_EVALUATOR_ABLATION,
            )
        )

        # ---------------------------------------------------------------
        # 4. LLG: global Zero-shot generation + Few-shot critic/reflection
        # ---------------------------------------------------------------
        low_level_goals, ll_score, ll_critique, ll_attempts, ll_history = run_top_down_reflection(
            target_type="Low Level Goals",
            call_function=generate_low_level_goals,
            define_args=(high_level_goals,),
            eval_mode=EvalMode.LOW_LEVEL,
            eval_args=(description, actors, high_level_goals),
            shotPromptingMode=LLG_PROMPTING_MODE,
            llama_ablation=LLG_EVALUATOR_ABLATION,
        )
        if not actors.actors or not high_level_goals.goals or not low_level_goals.low_level_goals:
            raise RuntimeError(
                "Top-down extraction produced an empty actor or goal collection; "
                "the previous output file was not overwritten."
            )

        output = {
            "name": dataset_name,
            "description": description,
            "rawDescription": raw_description,
            "documentationPreprocessing": preprocessing_info,
            "actors": [actor.name for actor in actors.actors],
            "highLevelGoals": [
                goal.description for goal in high_level_goals.goals
            ],
            "lowLevelGoals": [
                goal.description for goal in low_level_goals.low_level_goals
            ],
            "structuredActors": actors.model_dump(mode="json"),
            "structuredHighLevelGoals": high_level_goals.model_dump(mode="json"),
            "structuredLowLevelGoals": low_level_goals.model_dump(mode="json"),
            "baselineScores": {
                "actors": actors_score,
                "highLevelGoals": hl_score,
                "lowLevelGoals": ll_score,
            },
            "baselineAttempts": {
                "actors": actors_attempts,
                "highLevelGoals": hl_attempts,
                "lowLevelGoals": ll_attempts,
            },
            "baselineScoreHistory": {
                "actors": actors_history,
                "highLevelGoals": hl_history,
                "lowLevelGoals": ll_history,
            },
            "baselineCritiques": {
                "actors": actors_critique,
                "highLevelGoals": hl_critique,
                "lowLevelGoals": ll_critique,
            },
            "baselineConfiguration": {
                "implementation": "original top-down prompts; Qwen generator and local Ministral critic",
                "generatorModel": llm_clients.QWEN_GENERATOR_MODEL,
                "actorPromptingMode": ACTOR_PROMPTING_MODE.name,
                "actorEvaluatorEnabled": not ACTOR_EVALUATOR_ABLATION,
                "highLevelGoalPromptingMode": HLG_PROMPTING_MODE.name,
                "lowLevelGoalPromptingMode": LLG_PROMPTING_MODE.name,
                "highLevelGoalEvaluatorEnabled": not HLG_EVALUATOR_ABLATION,
                "lowLevelGoalEvaluatorEnabled": not LLG_EVALUATOR_ABLATION,
                "goalEvaluatorModel": llm_clients.configured_critic_model(),
                "qualityThreshold": TOP_DOWN_QUALITY_THRESHOLD,
                "maxReflectionAttempts": TOP_DOWN_MAX_ATTEMPTS,
                "evaluatorFewShotExampleCount": TOP_DOWN_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT,
                "evaluatorPromptStrategy": "original_prompt",
            },
            "pipelineConfiguration": {
                "actors": {
                    "promptingMode": ACTOR_PROMPTING_MODE.name,
                    "evaluator": not ACTOR_EVALUATOR_ABLATION,
                },
                "highLevelGoals": {
                    "promptingMode": HLG_PROMPTING_MODE.name,
                    "evaluator": not HLG_EVALUATOR_ABLATION,
                },
                "lowLevelGoals": {
                    "promptingMode": LLG_PROMPTING_MODE.name,
                    "evaluator": not LLG_EVALUATOR_ABLATION,
                    "generationStrategy": "all_hlgs_in_one_call",
                    "evaluationStrategy": "single_global_call_with_original_prompt",
                },
                "profile": PIPELINE_PROFILE,
            },
        }

        output_file = top_down_output_file_for(dataset_name)
        with output_file.open("w", encoding="utf-8") as file:
            json.dump(
                output,
                file,
                indent=4,
                ensure_ascii=False,
                default=str,
            )

        generated_descriptions[dataset_name] = description
        generated_actor_objects[dataset_name] = actors
        generated_hl_objects[dataset_name] = high_level_goals
        generated_ll_objects[dataset_name] = low_level_goals

        print(
            f"[{dataset_name}] saved: {output_file.name} | "
            f"Actors={len(actors.actors)} (Zero-shot, evaluator ON Few-shot) | "
            f"HLG={len(high_level_goals.goals)} (Zero-shot, evaluator ON Few-shot) | "
            f"LLG={len(low_level_goals.low_level_goals)} "
            f"(Zero-shot, evaluator ON Few-shot, single global call) | "
            f"evaluator={llm_clients.configured_critic_model()}"
        )

    except Exception as error:
        error_text = " ".join(str(error).split())
        if len(error_text) > 800:
            error_text = error_text[:797] + "..."
        baseline_errors[dataset_name] = (
            f"{type(error).__name__}: {error_text}"
        )
        print(
            f"[{dataset_name}] Top-down extraction failed: "
            f"{type(error).__name__}: {error_text}"
        )


for project_index, groundtruth in enumerate(GROUNDTRUTHS, start=1):
    print(
        f"\n=== PROJECT {project_index}/{len(GROUNDTRUTHS)}: "
        f"{groundtruth['name']} ==="
    )
    run_top_down_for_dataset(groundtruth)



=== PROJECT 1/12: SIA Project 25 26 ===
[SIA Project 25 26] Documentation ready | raw=5824 chars | pipeline=5824 chars
Actors STARTING... (attempt 1)
No feedback provided!
Actors DONE...
actors=[Actor(name='Citizen', description='Registered user who submits reports and receives updates about their reports.'), Actor(name='Municipal Administrator', description='Administrator who configures system settings and manages access for external companies.'), Actor(name='Organization Office Staff', description='Staff responsible for reviewing and approving citizen reports.'), Actor(name='Technical Office Staff', description='Staff responsible for assigning and managing reports based on category.'), Actor(name='External Maintenance Personnel', description='Maintenance workers from external companies who handle reports assigned to them.')]
Evaluation for Actors STARTING...
Evaluation for Actors DONE...
Score: 10.0
Critique: The actors defined are comprehensive and accurately represent the roles in

## 3. Riepilogo

Al termine viene mostrato il numero di dataset completati e falliti. Ogni dataset completato produce un JSON separato in `output/top_down/`.


In [4]:
completed_outputs = globals().get("generated_ll_objects", {})
execution_errors = globals().get("baseline_errors", {})
skipped_datasets = globals().get("skipped_outputs", {})
output_directory = globals().get("TOP_DOWN_DIR")

print("\nTop-down execution summary")
print("--------------------------")
print(f"Configured datasets: {len(GROUNDTRUTHS)}")
print(f"Completed datasets: {len(completed_outputs)}")
print(f"Failed datasets: {len(execution_errors)}")
print(f"Skipped datasets: {len(skipped_datasets)}")
for dataset_name, output_file in skipped_datasets.items():
    print(f"- {dataset_name}: kept {output_file.name}")
print(f"Output directory: {output_directory or 'pipeline not executed yet'}")

if execution_errors:
    print("Errors:")
    for dataset_name, error in execution_errors.items():
        print(f"- {dataset_name}: {error}")
elif not completed_outputs and not skipped_datasets:
    print("Pipeline not executed: run all previous cells first.")


Top-down execution summary
--------------------------
Configured datasets: 12
Completed datasets: 8
Failed datasets: 3
Skipped datasets: 1
- Recycling: kept Recycling_ZERO_SHOT_feedback_original_top_down_top_down.json
Output directory: C:\Users\agnes\Desktop\TESI FINALE CON EXPERIMENT\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\top_down
Errors:
- La Reine Marlene - SIA Project 23 24: RuntimeError: Qwen reached its output-token budget; no incomplete result was accepted. Increase QWEN_GENERATOR_MAX_TOKENS and the loaded context.
- Genome Nexus: RuntimeError: Qwen locale 'qwen/qwen3-vl-8b' su http://127.0.0.1:1234/v1: Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 129135>= n_ctx: 8192). Try to load the model with a larger context length, or provide a shorter input.'}
- Gestao Hospital: RuntimeError: Qwen locale 'qwen/qwen3-vl-8b' su http://127.0.0.1:1234/v1: Error code: 400 - {'error': 'The number of 